In [ ]:
from xgboost import XGBClassifier

In [ ]:
import pandas as pd
import sys
import os

if "Desktop" in os.getcwd():
    base_path = "."
else:
    base_path = "."

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import (
    Datasets,
    DATA_FOLDER_PREFIX,
    Dataset,
    NELA_SOURCE_LABELS,
)
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv, save_to_csv
from src.weak_labeling.consts.globals import NLP

from src.weak_labeling.consts.spacy_consts import (
    ADJECTIVE,
    NOUN,
    PRONOUN,
    PROPER_NOUN,
    NUMBER,
    CARDINAL_NUMBER,
    VERB,
    VERB_PRESENT_PARTICIPLE,
    VERB_PAST_PARTICIPLE,
    VERB_THIRD_PERSON,
    DETERMINER,
    ADPOSITION,
    ADVERB,
    COMPARATIVE_ADVERB,
    SUPERLATIVE_ADVERB,
    PERSONAL_PRONOUN,
    POSSESSIVE_PRONOUN,
    INTERJECTION,
    SYMBOL,
    WH_DETERMINER,
    MODAL,
    EXISTENTIAL,
    PUNCTUATION,
    PERIOD,
    QUESTION_MARK,
    COMMA,
    EXCLAMATION_POINT,
    COLON,
    SEMICOLON,
    PARENTHESES_OPEN,
    PARENTHESES_CLOSE,
    QUOTATION_MARK,
    SPACE,
)

# covid_df: pd.DataFrame = read_raw_data(Datasets.NELA_COVID_2021_DATASET)
# elections_df: pd.DataFrame = read_raw_data(Datasets.NELA_ELECTIONS_DATASET_SAMPLED)

In [ ]:
datasets = [
    Datasets.NELA_CAPITOL_2021,
    Datasets.NELA_COVID_2021,
    Datasets.NELA_COVID_2020,
    Datasets.NELA_ELECTIONS_2020,
    Datasets.NELA_ELECTIONS_SUBSET_2020,
]
for dataset in datasets:
    df: pd.DataFrame = read_csv(dataset.injected_path)
    print(min(df["label"].value_counts())*2)

In [ ]:
path = Datasets.NELA_ELECTIONS_2020.injected_path
injected_df: pd.DataFrame = read_csv(path)

In [ ]:
min(injected_df["label"].value_counts())

In [ ]:
injected_df.head()

In [ ]:
injected_df = injected_df.drop(
    [
        'model_xgboost_title_label',
        'model_xgboost_content_label',
        'model_adaboost_title_label',
        'model_adaboost_content_label',
        'model_logistic_regression_title_label',
        'model_logistic_regression_content_label',
        'model_random_forest_title_label',
        'model_random_forest_content_label',
        'model_naive_bayes_title_label',
        'model_naive_bayes_content_label',
    ],
    axis=1
)

In [ ]:
injected_df.head()

In [ ]:
save_to_csv(injected_df, path)

In [ ]:
for idx, row in covid_df.sample(n=1000).iterrows():
    print(row.content[:1000])
    print("\n\n\n")
    print("------------------------------")

In [ ]:
for idx, row in covid_df.sample(n=1000).iterrows():
    print(row.title[:1000])
    print("\n\n\n")
    print("------------------------------")

In [ ]:
sampled_df = covid_df.sample(n=1000)

In [ ]:
for idx, row in sampled_df.iterrows():
    print(row.content[:1000])
    print("\n\n\n")
    print("------------------------------")

In [ ]:
for i, row in sampled_df.iterrows():
    val = row["content"].encode("ascii", errors="ignore").decode()
    val =  val.replace(" @ @ @ @ @ @ @", "")
    val =  val.replace(" @ @ @ @ @", "")
    sampled_df.at[i,'content'] = val

In [ ]:
for idx, row in sampled_df.iterrows():
    print(row.content[:1000])
    print("\n\n\n")
    print("------------------------------")

In [ ]:

sampled_covid_df: pd.DataFrame = read_raw_data(Datasets.NELA_COVID_2021_DATASET_SAMPLED)

In [ ]:
for idx, row in sampled_covid_df.iterrows():
    if "@ @ @ @ @" in row.content:
        print(row.content)
        print(row.title)
        print(row.label)
        print("\n\n\n")
        print("------------------------------")

In [ ]:
sampled_covid_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

In [ ]:
print(sampled_elections_df.sample(n=10).head())

In [ ]:
print(sampled_elections_df.value_counts())

In [ ]:
sampled_elections_df.label.value_counts()

In [ ]:
def is_string_series(s : pd.Series):
    if isinstance(s.dtype, pd.StringDtype):
        # The series was explicitly created as a string series (Pandas>=1.0.0)
        return True
    elif s.dtype == 'object':
        # Object series, check each value
        return all((v is None) or isinstance(v, str) for v in s)
    else:
        return False

In [ ]:
print(is_string_series(covid_df.content))

In [ ]:
print(len(covid_df))
covid_df.dropna(inplace=True)
print(len(covid_df))

In [ ]:
for idx, row in covid_df.iterrows():
    if not isinstance(row.title, str):
        print(row.title)
        break

In [ ]:
print(covid_df.dtypes)
print(len(covid_df))

In [ ]:
from src.utils.preprocessing_utils import cleanup_df

print(len(covid_df))
covid_cleaned_df = cleanup_df(covid_df)
print(len(covid_cleaned_df))

In [ ]:
covid_cleaned_df['content_length']  = covid_cleaned_df['content'].str.len()

In [ ]:
covid_cleaned_df["content"].apply(len).mean()

In [ ]:
covid_cleaned_df.head()

In [ ]:
index = covid_cleaned_df.content_length.argmax()
covid_cleaned_df.iloc[index]

In [ ]:
covid_df_shorter = covid_cleaned_df[covid_cleaned_df['content'].apply(len) < 5000] 

In [ ]:
len(covid_df_shorter)

In [ ]:
covid_df_shorter.hist(column='content_length', bins=200);

In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

for idx, row in sampled_elections_df.iterrows():
#     print(row.content)
    doc = nlp(row.content)
#     for token in doc:
#         print(token.pos_)
#         print(token)
#         print(type(token))
#     words_lower = [word.text.lower() for word in doc]
#     unique_words = set(words_lower)
#     print(words_lower)
#     print(unique_words)
#     print(len(words_lower))
#     print(len(unique_words))
    sentence_lengths = []
    for sentence in doc.sents:
        sentence_lengths.append(len(sentence))
        print(sentence)
    print(sentence_lengths)
    break

In [ ]:
Datasets.NELA_COVID_2021.injected_path

In [ ]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)
# injected_df: pd.DataFrame = read_csv('temp/nela-covid-2021-sampled.csv')

In [ ]:
injected_df.label.value_counts()

In [ ]:
injected_df['content_flesch_kincaid_index'].plot(kind='kde')

In [ ]:
print(injected_df['content_flesch_kincaid_index'].min())
print(injected_df['content_flesch_kincaid_index'].max())

In [ ]:
text = sampled_df.sample(n=1).content.iloc[0]

In [ ]:
from readability import Readability

readability = Readability(text)

In [ ]:
!conda remove py-readability-metrics

In [ ]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_ELECTIONS_2020.injected_path)

In [ ]:
for col in injected_df.columns:
    count = (injected_df[col] == 0).sum()
    print(f"{col}: {count}")

In [ ]:
for idx, row in injected_df.iterrows():
    if QUESTION_MARK in row.content:
        print("found")
        print(row.content)
        print("\n--------------------\n")

        content_spacy = NLP(row.content)
        for token in content_spacy:
            # print(token.text)
            if token.text == QUESTION_MARK:
                print(token.pos_)
                print(token.tag_)
                print(token.text)

        break

In [ ]:
a_text = "This could work."
s_text = NLP(a_text)
for token in s_text:
    print(f"pos: {token.pos_}, tag: {token.tag_}")

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('distilbert-base-nli-mean-tokens')

sentences = [
    'the person wear red T-shirt',
    'this person is walking',
    'the boy wear red T-shirt'
    ]
sentence_embeddings = model.encode(sentences)

for sentence, embedding in zip(sentences, sentence_embeddings):
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print("")

In [ ]:
from sentence_transformers import util
print(util.pytorch_cos_sim(sentence_embeddings[0], sentence_embeddings[1]))
print(util.pytorch_cos_sim(sentence_embeddings[0], sentence_embeddings[2]))
print(util.pytorch_cos_sim(sentence_embeddings[1], sentence_embeddings[2]))

In [ ]:
title = "I am some other text!"
title_embedding = model.encode(title)
spacy_s = NLP("Hey there. How are you. Hope you are better")
similarities = []
for sentence in spacy_s.sents:
    print(sentence)
    current_sentence_embedding = model.encode(str(sentence))
    similarities.append(
        float(util.pytorch_cos_sim(title_embedding, current_sentence_embedding))
    )

In [ ]:
from statistics import mean

mean(similarities)